# =======
# Initializion
# =======

In [ ]:
import pandas as pd
import numpy as np
import regex as re
import os

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

import meteostat as ms
from datetime import datetime, timedelta
import helper_functions as f

# spark biblioteker fra labene
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first, input_file_name,
    regexp_extract
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType
)

In [5]:
# Create the SparkSession - the entry point for all Spark functionality
spark = SparkSession.builder \
    .appName("DAT535-Project-FootballStats") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Spark Version: 4.3.0
Application ID: local-1794409876380
Spark UI: http://10.255.255.254:4040


# ========
# Bronze layer
# ========

### OpenData Football data

Paths:

In [ ]:
COMPETITIONS_PATH = "../open-data/competitions.json"
MATCHES_PATH = "../open-data/data/events"
EVENTS_PATH = "../open-data/data/events"

# Weather PATH?

In [6]:
LA_LIGA_MATCHES_PATH = "../open-data/data/matches/11"

matches_df = spark.read.json(LA_LIGA_MATCHES_PATH, multiLine=True).select(
    col("match_id"),
    col("match_date"),
    col("kick_off"),
    col("stadium.name").alias("stadium")
)

In [7]:
EVENTS_PATH = "../open-data/data/events"

match_ids = matches_df.select(
    col("match_id")
).collect()

paths = []
for row in match_ids:
    paths.append(os.path.join(EVENTS_PATH, (str(row.match_id) + ".json")))

events_df = spark.read.json(paths, multiLine=True) \
                        .withColumn("match_id", 
                        regexp_extract(input_file_name(), r"([0-9]+).json", 1)) \
                        .select(
                            col("match_id"),
                            col("pass.height"),
                            col("pass.length"),
                            col("duration"),
                            col("pass.outcome")
                        )

events_df.write.mode("overwrite").parquet("bronze")

26/11/11 16:11:49 WARN MemoryManager: Total allocation exceeds 95.00% (2,040,109,440 bytes) of heap memory
Scaling row group sizes to 95.00% for 16 writers
26/11/11 16:11:51 WARN MemoryManager: Total allocation exceeds 95.00% (2,040,109,440 bytes) of heap memory
Scaling row group sizes to 95.00% for 16 writers


### Meteostat Weather data

In [8]:
# Bundling up the first and last match for doing fewer, but bigget API-calls
unique_stadiums_df = matches_df.groupBy("stadium") \
                            .agg(min("match_date").alias("first_match"), max("match_date").alias("last_match"))


In [16]:
from datetime import datetime

STADIUM = "Bernabéu"
lat, long = f.get_coordinates(STADIUM)
POINT = ms.Point(lat, long)
stations = ms.stations.nearby(POINT, limit=3)

unique_match_dates = matches_df.where(col("stadium") == STADIUM) \
                           .select("match_date", "kick_off") 

weather = []

for row in unique_match_dates.collect():
    kick_off = datetime.strptime(f"{row.match_date} {row.kick_off}", "%Y-%m-%d %H:%M:%S.%f") 
    ts = ms.hourly(stations, kick_off - timedelta(hours=3), kick_off + timedelta(hours=3))
    print(len(ts))
    weather.append(ms.interpolate(ts, POINT).fetch())

weather = pd.concat(ts)

10
1
1
10
10
10
1
9
10
10
2
12
10
1
9
12
7
11
6
3
6
13
2
5
3
2
4
6
2
3
4
7
5
0


ValueError: No objects to concatenate

26/11/12 13:11:23 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 36341355 ms exceeds timeout 120000 ms
26/11/12 13:11:23 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/12 13:11:27 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:143)
	at org.apache.spark.sto

In [ ]:
from datetime import date

for row in unique_stadiums_df.collect():
    
    lat, long = f.get_coordinates(row.stadium)

    print(row.first_match, type(row.first_match))
    # Creates a datetime from a string (e.g. datetime.striptime("2026-11-03 17:07:01.000", "%Y-%m-%d %H:%M:%S.%f"))
    first_match = date.strptime(row.first_match, "%Y-%m-%d")
    last_match = date.strptime(row.last_match, "%Y-%m-%d")
    POINT = ms.Point(lat, long)
    stations = ms.stations.nearby(POINT, limit=4)

    time_series = ms.hourly(stations, first_match, last_match)
    weather = ms.interpolate(time_series, POINT).fetch()
    break


2006-08-28 <class 'str'>


KeyboardInterrupt: 

# =======
# Silver layer
# =======

# =======
# Gold layer
# =======